## Public Key Encryption

A **public-key encryption** scheme $\mathcal{E}$ is a triple of efficient algorithm: a **key generation algorithm** $G$, an **encryption algorithm** $E$, a **decryption algorithm** $D$.


- $G$ is a *probabilistic algorithm* that is invoked ${\displaystyle ({\mathit {pk}},{\mathit {sk}})\xleftarrow{\text{R}} {G} ()}$
  
- $E$ is a *probabilistic algorithm* that is invoked ${\displaystyle {\mathit {c}} \xleftarrow{\text{}} {E(pk,m)}}$, where $m$ is a message and $c$ a ciphertext

- $D$ ia a *deterministic algorithm* that is invoked ${\displaystyle {\mathit {m}} \xleftarrow{\text{}} {D(sk,c)}}$

- We require correctness property, decryption undoes encryption $Pr[D(sk, E(pk,m)) = m] = 1$.  

- Messages are assumed to lie o finite **message space** $\mathcal{M}$ and ciphertext in some finite **ciphertext space** $\mathcal{C}$.



In [1]:
from sympy import isprime
from utils import *
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes

print("=" * 70)
print("✓ Environment ready")
print("=" * 70)

✓ Environment ready


## RSA

Vediamo le funzioni del PKE basato sul problema IFP.

#### Keygen

Le chiavi vengono generate nel seguente modo:

- Si scelgono due primi $p,q$ e si calcola $\;$ $n = pq$
- Si sceglie esponente di cifratura $e$ tale che $\;$ $gcd(e, \varphi(n)) = 1$
- Si sceglie l'esponente di decifratura $d$ tale che $\;$ $ed = 1 \ \text{mod} \ \varphi(n)$


$$pk = \{n,e\},\quad sk = \{d,q,p\}$$


#### Encryption

Dato un messaggio $m$ codificato come intero prima della cifratura (tipicamente convertendo la stringa in bytes e i bytes in un intero) tale che $0 \le m < n$.  
La cifratura con la chiave pubblica $\;$ $pk = \{n, e\}$ $\;$ è:

$$c = m^e \mod n$$


#### Decryption

Dato un ciphertext $c$, la decifratura con la chiave privata $\;$ $sk = \{d, n\}$ $\;$ è:

$$m = c^d \mod n$$


In [2]:
# Implementare RSA Keygen, Encryption, Decryption
def genera_primo(bit=1024):
    while True:
        n = random.getrandbits(bit)
        if isprime(n):
            return n

def inverso_moltiplicativo(a, p):
    old_r, r = a, p
    old_x, x = 1, 0
    while r != 0:
        q = old_r // r
        old_r, r = r, old_r - q * r
        old_x, x = x, old_x - q * x
    return old_x % p


# RSA KEYGEN
def rsa_keygen(bit=128, e=65537):
    """Genera una coppia di chiavi RSA. Ritorna (n, e, d, p, q)."""
    p = genera_primo(bit)
    q = genera_primo(bit)
    while q == p:
        q = genera_primo(bit)

    n = p * q
    phi = (p - 1) * (q - 1)
    d = inverso_moltiplicativo(e, phi)

    return {"n": n, "e": e, "d": d, "p": p, "q": q}

def rsa_encrypt(msg: str, e: int, n: int) -> int:
    """Cifra una stringa con la chiave pubblica (e, n)."""
    m = int.from_bytes(msg.encode('utf-8'), byteorder='big')
    if m >= n:
        raise ValueError(f"Messaggio troppo lungo: {m.bit_length()} bit >= n ({n.bit_length()} bit)")
    return pow(m, e, n)


def rsa_decrypt(c: int, d: int, n: int) -> str:
    """Decifra un intero con la chiave privata (d, n)."""
    m = pow(c, d, n)
    num_bytes = (m.bit_length() + 7) // 8
    return m.to_bytes(num_bytes, byteorder='big').decode('utf-8')

chiavi = rsa_keygen()
n = chiavi["n"]
e = chiavi["e"]
d = chiavi["d"]
p = chiavi["p"]
q = chiavi["q"]

msg = "ciao"
c = rsa_encrypt(msg, e, n)
m = rsa_decrypt(c, d, n)

print(m)

ciao



## Semantic Security

Una proprieta' fondamentale per gli schemi di Public Key Encryption e' la **semantic security**.  
> A partire dal ciphertext si possono recuperare solo informazioni trascurabili sul plaintext; un messaggio cifrato non deve rilevare alcuna informazione utile sul messaggio originale.


La nozione di **semantic security** si puo' tradurre in un attack game (modella solo eavesdropping).   
Invece di chiedere all'attaccante "hai imparato qualcosa sul messaggio?", gli facciamo affrontare una scelta tra due possibili messaggi.  
L'obiettivo del gioco e' verificare che l'attaccante $\mathcal{A}$ non sia in grado di comprendere quale plaintext corrisponde al ciphertext che ha ricevuto.

<br>

<div style="display:flex; text-align:center; justify-content: center; gap: 30px;">
  <div style="width: 500px; text-align: center;">
    <img src="imgs/SemanticSecurity.svg" style="width: 500px; height: auto;">
  </div>

</div>


In [61]:
# Verificare se RSA e' semantically secure

keys = rsa_keygen()
n = chiavi["n"]
e = chiavi["e"]
d = chiavi["d"]
p = chiavi["p"]
q = chiavi["q"]

m0 = "ciao"
m1 = "come stai"

print(f"Adversary has chosen: \n m0 = {m0} \n m1 = {m1}")
print()

c0 = rsa_encrypt(m0, e, n)
c1 = rsa_encrypt(m1, e, n)

print(f"Adversary known pk so can encrypt: \n c0 = {c0} \n c1 = {c1}")
print()

b = random.randint(0,1)

if b == 0:
    cb = rsa_encrypt(m0, e, n) 
else:
    cb = rsa_encrypt(m1, e, n) 

print(f"Challenger output: \n cb = {cb}")

if cb == c0:
    b_hat = 0
elif cb == c1:
    b_hat = 1

print()
print(f"Adversary output: b_hat = {b_hat}")
print(f"Challenger has chosen: b = {b}")
print() 

if b_hat == b:
    print("Adversary wins!")
else:
    print("Adversary loses!")


Adversary has chosen: 
 m0 = ciao 
 m1 = come stai

Adversary known pk so can encrypt: 
 c0 = 6587710397262066443640752669230119269920897299465597158270320931118957041776 
 c1 = 8835081039441011871570852949181483329407936357541907260085372163436594294421

Challenger output: 
 cb = 8835081039441011871570852949181483329407936357541907260085372163436594294421

Adversary output: b_hat = 1
Challenger has chosen: b = 1

Adversary wins!


### Relazione con IND-CPA Security

In PKE la semantic security implica la **semantic security against chosen plaintext attack**.  
Questo appunto perché l'avversario può scegliere plaintext e ottenere le relative cifrature, dato che basta solo la chiave pubblica.  

La differenz asostanzaile con l'attack game con IND-CPA e' che un attaccante puo' fare $\mathcal{Q}$ query

## OAEP
Plain RSA is not semanmtically secure because the **Encryption Algorithm** is determinist and not probabilistic.  
Grazie a questa conversione diventa probabilistico e anche IND-CCA2



<div style="display:flex; text-align:center; justify-content: center; gap: 30px;">
  <div style="width: 500px; text-align: center;">
    <img src="imgs/OAEP.svg" style="width: 500px; height: auto;">
  </div>

</div>

In [60]:
# Parametri
K = 1024
K_BYTES = 1024 // 8
K0_BYTES = 128 // 8  # 16 byte
K1_BYTES = 128 // 8  # 16 byte
N_MAX = K_BYTES - K0_BYTES - K1_BYTES

# Funzioni G e H disponibli
# G(seed)
# H(msg_padded)

def oaep_encode(msg):
    msg_bytes = msg.encode('utf-8')
    if len(msg_bytes) > N_MAX:
        print("Spezzare")
    # MSG TO INT AND PADDING
    msg_padded = (
        msg_bytes
        + b'\x00' * (N_MAX - len(msg_bytes))
        + b'\x00' * K1_BYTES
    )
    print("=" * 70)
    print("Messaggio Paddato")
    print("=" * 70)
    print(msg_padded)
    r = secrets.token_bytes(K0_BYTES)
    # OUTPUT 
    X = xor_bytes(msg_padded, G(r))
    Y = xor_bytes(r, H(X))
    em = X + Y
    return int.from_bytes(em ,byteorder='big')



msg = "ciao sono"

print("=" * 70)
print("Messaggio Originale")
print("=" * 70)
print(msg.encode('utf-8'))
print()

msg_1 = oaep_encode(msg)
print() 

print("=" * 70)
print("Messaggio dopo OAEP")
print("=" * 70)
print(msg_1)

Messaggio Originale
b'ciao sono'

Messaggio Paddato
b'ciao sono\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

Messaggio dopo OAEP
53336655585876093765435223018540645865665186343417636151920154778375054787383632259886874670442890719781399851352326377798813750539418922271636356786716234360892373777534806909454333078160676552136062230931143695758289383361511131487329764412929670104896893693279780758143557852580635999199971605584967209380


## RSA con libreria

https://cryptography.io/en/stable/hazmat/primitives/asymmetric/rsa

In [65]:

private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

public_key = private_key.public_key()

print("=" * 70)
print("RSA - cryptography")
print("=" * 70)

print("Chiave privata generata")
print("Dimensione:", private_key.key_size, "bit")

print("Chiave pubblica generata")
print()


# =========================
# 2. Messaggio
# =========================

message = b"ciao sono padre pio"

print("Messaggio originale:")
print(message)
print()


# =========================
# 3. Cifratura con RSA-OAEP
# =========================

ciphertext = public_key.encrypt(
    message,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print("Ciphertext:")
print(ciphertext)
print()


# =========================
# 4. Decifratura
# =========================

plaintext = private_key.decrypt(
    ciphertext,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print("Messaggio decifrato:")
print(plaintext)
print()


# =========================
# 5. Verifica
# =========================

print("Messaggio originale == messaggio decifrato:",
      message == plaintext)

RSA - cryptography
Chiave privata generata
Dimensione: 2048 bit
Chiave pubblica generata

Messaggio originale:
b'ciao sono padre pio'

Ciphertext:
b"\td^n\xee\xe7B>\xd2\x05\r\x0f\x1a\xf2\xa1\xf8\xaf\xda\x9d@iF \xd5=B_\x8c\xbbm\x9a;H\xe0\x07F\xf2-fU\x1aa\xa1\xd3d\x9c\xab\xe5\x0e\xd2\xa6\xe4\xfe\xf3N\xb3\xfc\xb8\xfa&\xe7\xeb)\x077^\xa4\xcfE\xfe\xef.\xb8\xed\x95\xe6z\xed\xad/\xb07\xa1&&\xb0\x96\xa6X\x85b<\x16N\xd4\xbf\xd0\xd8\xbc#\x10\x95<O\x08\xb8\xa6\xc1\xe1\xb6)\xda?8j\\\xdd\xe1\xeb\x8c\xa9F}\xef\x9c\xf0d\x7f\x87\xb3\xe6K\x9d\xb0i\x95\xbc\x97\xf8af:\xd14q\x0c2\x18\xa6\x8b\xe4=I@\xa2[I\x1dK\x02\xb3\xcf\x0c\x89`\xee\xb9\x17\xb5V\xbc\xcb*[\xc1\xcc\xee\xe2\xe0\x84y\x89\xf2\x9c\x9f\x12\xdd\x13?\xf9\x81\x05\xe0}\x84\xf7\xea\x91\xc0F\x05\xeb\xb47>\x14\x06s\xb7|5\xccZ\x14\x04f\xc5'\xce\x15\xff\xa4b\x028|\x01\xdd\x8e!Yo\xb3\xab1\x1b\x86l\xd02\xbc\x08r\x08K;\xdc\xee\xf3\x8dsO\xca\xeeA7"

Messaggio decifrato:
b'ciao sono padre pio'

Messaggio originale == messaggio decifrato: True
